# Track A — S24 SD LoRA VS Generation (Colab)

**전제조건**: 런타임 유형 → GPU (T4 이상)

**순서**: [1] → [2] → [3] → [4] → [5] → [6] → [7]

In [ ]:
# [1] GPU 확인
import torch
assert torch.cuda.is_available(), 'GPU 없음: 런타임 > 런타임 유형 변경 > T4 GPU'
print(f'torch  {torch.__version__}')
print(f'GPU    {torch.cuda.get_device_name(0)}')

In [ ]:
# [2] 환경 세팅
# Colab 환경: torch 2.11.0+cu130 / torchvision 0.26.0+cu128 / torchaudio cu128
# torchvision/torchaudio 모두 cu128 빌드 → cu130 torch와 CUDA major 불일치
# 세 라이브러리 모두 CUDA 버전 체크만 비활성화하면 실제 연산은 정상 동작
import subprocess, os, re

# ── 1. torchvision CUDA 버전 체크 비활성화 ───────────────────────────────────
_ext = '/usr/local/lib/python3.12/dist-packages/torchvision/extension.py'
with open(_ext) as f: _s = f.read()
_s = _s.replace('def # _check_cuda_version()  # patched:', 'def _check_cuda_version():')
_s = re.sub(r'if t_major != tv_major:', 'if False:  # patched', _s)
_s = re.sub(r'^_check_cuda_version\(\)\s*$', '# _check_cuda_version()  # patched', _s, flags=re.MULTILINE)
with open(_ext, 'w') as f: f.write(_s)
print('torchvision patched')

# ── 2. torchaudio CUDA 버전 체크 비활성화 (동일 문제) ────────────────────────
_ta = '/usr/local/lib/python3.12/dist-packages/torchaudio/_extension/utils.py'
if os.path.exists(_ta):
    with open(_ta) as f: _s = f.read()
    _s = re.sub(r'if ta_version != t_version:', 'if False:  # patched', _s)
    with open(_ta, 'w') as f: f.write(_s)
    print('torchaudio patched')
else:
    print('torchaudio: not found, skipping')

# ── 3. peft / diffusers / accelerate 설치 (--no-deps: torch 버전 유지) ───────
subprocess.run(['pip', 'install', '-q', '--no-deps',
                'peft', 'diffusers', 'accelerate'], check=True)

# ── 4. transformers + 의존성 설치 ─────────────────────────────────────────────
subprocess.run(['pip', 'install', '-q', '-U',
                'transformers', 'tokenizers', 'huggingface-hub', 'safetensors',
                'packaging', 'h5py', 'regex', 'tqdm', 'filelock', 'requests'], check=True)

# ── 5. 검증 (fresh subprocess — 현재 커널 import 상태와 무관하게 확인) ─────────
r = subprocess.run(['python', '-c', '''
import torch, torchvision, torchaudio
from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler
from peft import LoraConfig, get_peft_model
import transformers
print(f"torch        {torch.__version__}")
print(f"torchvision  {torchvision.__version__}")
print(f"torchaudio   {torchaudio.__version__}")
print(f"diffusers    OK")
print(f"peft         OK")
print(f"transformers {transformers.__version__}")
print("ALL OK")
'''], capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print('=== ERROR ===')
    print(r.stderr[-3000:])
    raise RuntimeError('환경 검증 실패 — 위 에러 확인 필요')

In [ ]:
# [3] 코드 clone / pull (S24 .npz 포함)
import os, subprocess

REPO    = 'https://github.com/heegyukim4043/test_diffusion_EEG_visualstim_image.git'
WORKDIR = '/content/vsvi_project'

def _clone():
    subprocess.run(['git', 'clone', REPO, WORKDIR], check=True)

if os.path.exists(WORKDIR):
    if not os.path.exists(os.path.join(WORKDIR, '.git')):
        subprocess.run(['rm', '-rf', WORKDIR], check=True)
        _clone()
    else:
        r = subprocess.run(['git', '-C', WORKDIR, 'pull', 'origin', 'main'])
        if r.returncode != 0:
            subprocess.run(['rm', '-rf', WORKDIR], check=True)
            _clone()
else:
    _clone()

os.chdir(WORKDIR)
subprocess.run(['git', 'log', '--oneline', '-3'])

npz = [f for f in os.listdir('preproc_vs_re') if 'subj_24' in f and f.endswith('.npz')]
ckpt = 'checkpoints_vsre_dino/20260604_091352_ch32_merged_ep200_supcon/subj24_best.pt'
print(f'S24 .npz: {len(npz)}개,  SupCon ckpt: {os.path.exists(ckpt)}')

In [ ]:
# [4] Google Drive 마운트 (체크포인트 저장)
from google.colab import drive
drive.mount('/content/drive')

CKPT_DRIVE = '/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen'
os.makedirs(CKPT_DRIVE, exist_ok=True)

dst = '/content/vsvi_project/checkpoints_vsre_lora_gen'
if not os.path.exists(dst):
    os.symlink(CKPT_DRIVE, dst)
print(f'checkpoints_vsre_lora_gen -> {CKPT_DRIVE}')

In [ ]:
# [5] 최종 확인
import os, torch
W = '/content/vsvi_project'
checks = {
    'CUDA':        torch.cuda.is_available(),
    'npz 10개':    len([f for f in os.listdir(f'{W}/preproc_vs_re') if 'subj_24' in f and f.endswith('.npz')]) == 10,
    'SupCon ckpt': os.path.exists(f'{W}/checkpoints_vsre_dino/20260604_091352_ch32_merged_ep200_supcon/subj24_best.pt'),
    'images':      os.path.exists(f'{W}/preproc_data_vi/images/01.png'),
    'ckpt_root':   os.path.islink(f'{W}/checkpoints_vsre_lora_gen') or os.path.isdir(f'{W}/checkpoints_vsre_lora_gen'),
}
for k, v in checks.items():
    print(f'  [{"OK" if v else "FAIL"}] {k}')
assert all(checks.values()), 'FAIL 항목 해결 후 재실행'

In [ ]:
# [6] S24 LoRA r=16 학습 (T4 기준 약 8~12시간)
import subprocess, os
os.chdir('/content/vsvi_project')
result = subprocess.run([
    'python', '-u', 'train_vs_re_lora_gen.py',
    '--subject_ids', '24',
    '--lora_r', '16',
    '--n_eeg_tokens', '16',
    '--epochs', '100',
    '--img_root', 'preproc_data_vi/images',
    '--supcon_ckpt', 'checkpoints_vsre_dino/20260604_091352_ch32_merged_ep200_supcon',
    '--ckpt_root', 'checkpoints_vsre_lora_gen',
])
print(f'Exit code: {result.returncode}')

In [ ]:
# [7] S24 LoRA r=32 학습 (r=16 완료 후 실행)
import subprocess, os
os.chdir('/content/vsvi_project')
result = subprocess.run([
    'python', '-u', 'train_vs_re_lora_gen.py',
    '--subject_ids', '24',
    '--lora_r', '32',
    '--n_eeg_tokens', '16',
    '--epochs', '100',
    '--img_root', 'preproc_data_vi/images',
    '--supcon_ckpt', 'checkpoints_vsre_dino/20260604_091352_ch32_merged_ep200_supcon',
    '--ckpt_root', 'checkpoints_vsre_lora_gen',
])
print(f'Exit code: {result.returncode}')

In [ ]:
# [8] 결과 확인
import glob
csvs = glob.glob('/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/**/results_dual_gallery.csv', recursive=True)
for p in sorted(csvs):
    print(p)
    with open(p) as f: print(f.read())